In [9]:
# AI Music Generator Using Genetic Algorithm
## Overview
# This project generates melodies using a Genetic Algorithm by evolving a population of musical note sequences over multiple generations.
# The generated melodies are exported as MIDI files and evaluated using fitness functions.

In [10]:
## Project Information
# **Language:** Python
# **Platform:** Google Colab
# **Algorithm:** Genetic Algorithm
# **Output:** MIDI Music Generation

In [11]:
# Install dependencies
!apt-get update -qq
!apt-get install -y fluidsynth ffmpeg
!pip install -q pydub

# Download SoundFont
!wget -q -O TimGM6mb.sf2 https://archive.org/download/free-soundfonts/TimGM6mb.sf2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  at-spi2-core fluid-soundfont-gm gsettings-desktop-schemas libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libdouble-conversion3 libevdev2
  libfluidsynth3 libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0
  libinput-bin libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5
  librsvg2-common libwacom-bin libwacom-common libwacom9 libxcb-icccm4
  libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 

In [12]:
!pip install mido

In [13]:
# AI Melody Improviser – Single Target, Multi-Style, Multi-Run, PDF Report
# ------------------------------------------------------------------------
# Improvements in this version:
# - Accepts SPACE or COMMA separated input (e.g., "C D E" or "C,D,E")
# - Classical fitness emphasizes pitch-contour matching (no more flat lines)
# - Per-style GA hyperparams (Classical explores more)
# - Replaces emoji in report with Unicode music symbols (no font warnings)
# - Adds a Summary Comparison TABLE (avg fitness, best, tempo, scale)
# - Minor polish on plots & PDF pages

import os
import random
from datetime import datetime
from statistics import mean, pstdev

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mido import Message, MidiFile, MidiTrack, bpm2tempo, MetaMessage

# -----------------------------
# Configuration & Dictionaries
# -----------------------------
SCALE_NOTES = {
    "C major": ['C', 'D', 'E', 'F', 'G', 'A', 'B'],
    "A minor": ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    "C minor": ['C', 'D', 'Eb', 'F', 'G', 'Ab', 'Bb'],
}

NOTE_TO_MIDI = {
    'C': 60, 'C#': 61, 'Db': 61,
    'D': 62, 'D#': 63, 'Eb': 63,
    'E': 64, 'F': 65, 'F#': 66, 'Gb': 66,
    'G': 67, 'G#': 68, 'Ab': 68,
    'A': 69, 'A#': 70, 'Bb': 70,
    'B': 71
}
MIDI_TO_NOTE = {v: k for k, v in NOTE_TO_MIDI.items()}

# Default GA hyperparameters (can be overridden by input)
POP_SIZE = 60
GENERATIONS = 120
MUT_RATE = 0.25
ELITE_FRAC = 0.5  # top 50% survive per generation

# Styles & defaults
STYLE_TEMPOS = {"soft": 90, "rap": 120, "classical": 72}
STYLE_SCALES = {"soft": "C major", "rap": "C major", "classical": "C major"}

# Optional style icons (use Unicode that renders in PDF)
STYLE_ICON = {"soft": "♪", "rap": "♬", "classical": "♫"}

# -----------------------------
# Utility
# -----------------------------
def parse_user_melody(s: str):
    """Accept 'C D E' or 'C,D,E' or mixed separators."""
    s = s.replace(",", " ")
    tokens = [tok.strip() for tok in s.split() if tok.strip()]
    return tokens

def sanitize_notes(seq, allowed_notes):
    """Replace out-of-scale notes with nearest allowed note by MIDI distance."""
    def nearest_allowed(note):
        if note in allowed_notes:
            return note
        m = NOTE_TO_MIDI[note]
        best = min(allowed_notes, key=lambda n: abs(NOTE_TO_MIDI[n] - m))
        return best
    return [nearest_allowed(n) for n in seq]

def melody_to_midi(melody, filename, tempo_bpm=100, ticks_per_beat=480, velocity=80, beat_div=1):
    """Save a monophonic melody to MIDI."""
    mid = MidiFile()
    mid.ticks_per_beat = ticks_per_beat
    track = MidiTrack()
    mid.tracks.append(track)
    track.append(MetaMessage('set_tempo', tempo=bpm2tempo(tempo_bpm), time=0))
    dur = int(ticks_per_beat * beat_div)
    for note in melody:
        nn = NOTE_TO_MIDI.get(note, 60)
        track.append(Message('note_on', note=nn, velocity=velocity, time=0))
        track.append(Message('note_off', note=nn, velocity=velocity, time=dur))
    mid.save(filename)

def coerce_to_length(seq, target_len, pad_with=None):
    if len(seq) == target_len:
        return seq[:]
    if len(seq) > target_len:
        return seq[:target_len]
    if pad_with is None:
        pad_with = seq[-1] if seq else 'C'
    return seq + [pad_with] * (target_len - len(seq))

# -----------------------------
# Fitness helpers
# -----------------------------
def interval(a, b):
    return abs(NOTE_TO_MIDI[a] - NOTE_TO_MIDI[b])

def contour(seq):
    """Return +1 / 0 / -1 contour steps by MIDI difference sign."""
    diffs = []
    for i in range(len(seq) - 1):
        d = NOTE_TO_MIDI[seq[i+1]] - NOTE_TO_MIDI[seq[i]]
        diffs.append(1 if d > 0 else (-1 if d < 0 else 0))
    return diffs

def contour_match_score(melody, target):
    """Reward matching rise/fall direction with target."""
    cm = contour(melody)
    ct = contour(target)
    L = min(len(cm), len(ct))
    score = 0
    for i in range(L):
        if cm[i] == ct[i]:
            score += 2          # strong reward for same direction
        elif cm[i] == 0 and ct[i] == 0:
            score += 1          # both hold
        elif cm[i] == 0 or ct[i] == 0:
            score += 0          # neutral if one holds
        else:
            score -= 1          # penalize opposite direction
    return score

# -----------------------------
# Fitness Functions by Style
# -----------------------------
def fitness_soft(melody, target):
    """Close to target + smooth, stepwise motion (small intervals)."""
    score = 0
    score -= sum(abs(NOTE_TO_MIDI[m] - NOTE_TO_MIDI[t]) for m, t in zip(melody, target))
    for i in range(1, len(melody)):
        d = interval(melody[i-1], melody[i])
        score += max(0, 7 - d)  # reward small steps (0-6 semitones)
    # light repetition bonus
    for i in range(len(melody) - 1):
        if melody[i] == melody[i+1]:
            score += 0.5
    return score

def fitness_rap(melody, target):
    """Close to target + encourage variety and some leaps (contrasting shapes)."""
    score = 0
    score -= sum(abs(NOTE_TO_MIDI[m] - NOTE_TO_MIDI[t]) for m, t in zip(melody, target))
    score += len(set(melody)) * 2
    # leap/step contrast bonus
    for i in range(2, len(melody)):
        d1 = interval(melody[i-2], melody[i-1])
        d2 = interval(melody[i-1], melody[i])
        if (d1 >= 4 and d2 <= 2) or (d1 <= 2 and d2 >= 4):
            score += 2
    return score

def fitness_classical(melody, target):
    """
    Classical: close to target + strong contour match + penalize large leaps;
    reward small stepwise motion and simple motifs. This prevents flat-lines.
    """
    score = 0
    # pitch proximity
    score -= sum(abs(NOTE_TO_MIDI[m] - NOTE_TO_MIDI[t]) for m, t in zip(melody, target))
    # contour alignment (big positive driver)
    score += 3.0 * contour_match_score(melody, target)
    # stepwise preference / large leap penalty
    for i in range(1, len(melody)):
        d = interval(melody[i-1], melody[i])
        score += max(0, 6 - d)      # reward small steps
        score -= max(0, d - 5) * 1  # penalize big jumps
    # tiny motif reward (ABA, repeated bigrams)
    for i in range(len(melody) - 2):
        if melody[i] == melody[i+2]:
            score += 1.0
        if i < len(melody) - 3 and melody[i:i+2] == melody[i+2:i+4]:
            score += 1.5
    return score

STYLE_FITNESS = {
    "soft": fitness_soft,
    "rap": fitness_rap,
    "classical": fitness_classical,
}

# Style-specific GA tuning
STYLE_MUT_RATE = {"soft": 0.22, "rap": 0.25, "classical": 0.35}
STYLE_ELITE_FRAC = {"soft": 0.5, "rap": 0.5, "classical": 0.45}

# -----------------------------
# GA Core
# -----------------------------
def random_melody(length, allowed_notes):
    return [random.choice(allowed_notes) for _ in range(length)]

def crossover(parent1, parent2):
    if len(parent1) < 2:
        return parent1[:]
    point = random.randint(1, len(parent1) - 1)
    return parent1[:point] + parent2[point:]

def mutate(melody, allowed_notes, rate=MUT_RATE):
    return [random.choice(allowed_notes) if random.random() < rate else n for n in melody]

def run_ga_once(target, allowed, style,
                pop_size=POP_SIZE, gens=GENERATIONS,
                mut_rate=MUT_RATE, elite_frac=ELITE_FRAC):
    fitness_fn = STYLE_FITNESS[style]
    length = len(target)
    population = [random_melody(length, allowed) for _ in range(pop_size)]

    best_hist, avg_hist = [], []

    for _ in range(gens):
        scores = [fitness_fn(m, target) for m in population]
        ranked = sorted(zip(scores, population), key=lambda x: x[0], reverse=True)
        best_hist.append(ranked[0][0])
        avg_hist.append(sum(scores) / len(scores))

        elites_n = max(2, int(elite_frac * pop_size))
        survivors = [m for _, m in ranked[:elites_n]]

        children = []
        while len(survivors) + len(children) < pop_size:
            p1, p2 = random.sample(survivors, 2)
            child = crossover(p1, p2)
            child = mutate(child, allowed, rate=mut_rate)
            children.append(child)

        population = survivors + children

    best_melody = max(population, key=lambda m: fitness_fn(m, target))
    best_score = fitness_fn(best_melody, target)
    return best_melody, best_score, best_hist, avg_hist

def run_ga_multi(target, allowed, style, runs=5):
    bests, best_scores, best_traces, avg_traces = [], [], [], []
    for _ in range(runs):
        m, s, best_hist, avg_hist = run_ga_once(
            target, allowed, style,
            pop_size=POP_SIZE,
            gens=GENERATIONS,
            mut_rate=STYLE_MUT_RATE.get(style, MUT_RATE),
            elite_frac=STYLE_ELITE_FRAC.get(style, ELITE_FRAC),
        )
        bests.append(m)
        best_scores.append(s)
        best_traces.append(best_hist)
        avg_traces.append(avg_hist)

    max_len = max(len(t) for t in best_traces)
    def pad_curve(c): return c + [c[-1]] * (max_len - len(c))
    best_traces = np.array([pad_curve(c) for c in best_traces])
    avg_traces  = np.array([pad_curve(c) for c in avg_traces])

    best_mean = best_traces.mean(axis=0)
    best_std  = best_traces.std(axis=0)
    avg_mean  = avg_traces.mean(axis=0)
    avg_std   = avg_traces.std(axis=0)

    idx = int(np.argmax(best_scores))
    best_melody = bests[idx]
    best_score  = best_scores[idx]

    return {
        "best_melody": best_melody,
        "best_score": best_score,
        "best_traces_mean": best_mean,
        "best_traces_std": best_std,
        "avg_traces_mean": avg_mean,
        "avg_traces_std": avg_std,
        "all_best_scores": best_scores,
    }

# -----------------------------
# Plotting
# -----------------------------
def plot_fitness_band(ax, mean_curve, std_curve, label):
    x = np.arange(1, len(mean_curve) + 1)
    ax.plot(x, mean_curve, label=label)
    ax.fill_between(x, mean_curve - std_curve, mean_curve + std_curve, alpha=0.2)

def plot_pitch_contour(ax, target, generated, title):
    ax.plot(range(len(target)), [NOTE_TO_MIDI[n] for n in target], marker='o', linestyle='-', label='Target')
    ax.plot(range(len(generated)), [NOTE_TO_MIDI[n] for n in generated], marker='x', linestyle='--', label='Generated')
    ax.set_title(title)
    ax.set_xlabel("Note Index")
    ax.set_ylabel("MIDI Note")
    ax.grid(True)
    ax.legend()

def page_title_fig(text_lines, fontsize=16):
    fig = plt.figure(figsize=(8.27, 11.69))  # A4 portrait
    fig.text(0.1, 0.9, text_lines[0], fontsize=fontsize+4, weight='bold')
    y = 0.84
    for line in text_lines[1:]:
        fig.text(0.1, y, line, fontsize=fontsize)
        y -= 0.04
    return fig

def monospaced_text_page(block_text):
    """Render a monospaced text block nicely on a PDF page (no emoji)."""
    # swap any leftover emoji with Unicode symbols that render
    block_text = (block_text
                  .replace("🎵", "♪")
                  .replace("🎶", "♫")
                  .replace("✅", "✓"))
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.06, 0.94, "Run Log", fontsize=18, weight='bold')
    y = 0.90
    for line in block_text.splitlines():
        fig.text(0.06, y, line, fontsize=11, family='monospace')
        y -= 0.028
        if y < 0.05:
            break
    return fig

def summary_table_page(rows, title="Summary Comparison"):
    """
    rows: list of [Style, Avg Fitness, Best Fitness, Tempo, Scale]
    """
    fig, ax = plt.subplots(figsize=(8.27, 11.69))
    ax.axis('off')
    ax.set_title(title, fontsize=18, pad=20, loc='left')
    col_labels = ["Style", "Avg. Fitness", "Best Fitness", "Tempo (BPM)", "Scale"]
    the_table = ax.table(cellText=rows, colLabels=col_labels, cellLoc='center', loc='upper left')
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(11)
    the_table.scale(1, 1.4)
    return fig

# -----------------------------
# Main CLI & Report
# -----------------------------
def main():
    global GENERATIONS, POP_SIZE

    print("\n=== AI Melody Improviser – Single Target, Multi-Style Report ===\n")

    # 1) Target melody
    user_melody_str = input("Enter target melody (space- or comma-separated, e.g. C C G G A A G F F E E D D C): ").strip()
    user_melody = parse_user_melody(user_melody_str)

    # Validate notes
    bad = [n for n in user_melody if n not in NOTE_TO_MIDI]
    if bad:
        print(f"\nSome notes not recognized: {bad}")
        print("Supported notes: C C# Db D D# Eb E F F# Gb G G# Ab A A# Bb B")
        return

    # 2) Runs / GA hyperparams
    try:
        runs = int(input("Number of GA runs per style (e.g., 5 or 10): ").strip())
    except:
        runs = 5
    try:
        gens = int(input(f"Generations per run (default {GENERATIONS}): ").strip() or GENERATIONS)
    except:
        gens = GENERATIONS
    try:
        pop = int(input(f"Population size (default {POP_SIZE}): ").strip() or POP_SIZE)
    except:
        pop = POP_SIZE

    GENERATIONS = gens
    POP_SIZE = pop

    # 3) Tempo overrides
    print("\n(Press Enter to keep defaults) Tempo defaults -> Soft: 90, Rap: 120, Classical: 72")
    tempo_overrides = {}
    for s in ["soft", "rap", "classical"]:
        t = input(f"Tempo for {s} BPM (default {STYLE_TEMPOS[s]}): ").strip()
        tempo_overrides[s] = int(t) if t.isdigit() else STYLE_TEMPOS[s]

    # 4) Optional: choose scale (default per style: C major)
    print("\nAvailable scales:", ", ".join(SCALE_NOTES.keys()))
    scale_input = input("Scale to use for ALL styles (blank = per-style defaults): ").strip()
    if scale_input and scale_input not in SCALE_NOTES:
        print("Scale not found; using per-style defaults.")
        scale_input = ""

    styles = ["soft", "rap", "classical"]
    now_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = f"aimelody_report_{now_stamp}"
    os.makedirs(out_dir, exist_ok=True)

    # -----------------------------
    # Run GA for each style (same target)
    # -----------------------------
    full_log_lines = []
    results = {}

    for style in styles:
        scale = scale_input if scale_input else STYLE_SCALES[style]
        allowed_notes = SCALE_NOTES[scale]

        # Constrain the target to scale for that style
        target = sanitize_notes(user_melody, allowed_notes)

        # Multi-run GA
        print(f"\n>>> Running {runs} GA runs for {style.title()} Style...")
        res = run_ga_multi(target, allowed_notes, style, runs=runs)
        results[style] = {
            "scale": scale,
            "tempo": tempo_overrides[style],
            "target": target,
            **res
        }

        # Save MIDI files & build log text
        time_tag = datetime.now().strftime("%H%M%S")
        gen_name = f"{style.title()}_generated_{time_tag}.mid"
        gen_path = os.path.join(out_dir, gen_name)
        target_name = f"{style.title()}_target_{time_tag}.mid"
        target_path = os.path.join(out_dir, target_name)

        melody_to_midi(target, target_path, tempo_bpm=tempo_overrides[style])
        melody_to_midi(res["best_melody"], gen_path, tempo_bpm=tempo_overrides[style])

        icon = STYLE_ICON.get(style, "")
        block = []
        block.append(f"{icon} Running GA for {style.title()} Style...")
        block.append(f"Saved {style.title()} generated melody to {gen_name}")
        block.append(f"{STYLE_ICON.get(style,'')} Target Melody: {results[style]['target']}")
        block.append(f"{STYLE_ICON.get(style,'')} Best Generated Melody: {res['best_melody']}")
        block.append(f"✓ Fitness Score: {res['best_score']}")
        full_log_lines.append("\n".join(block))

        # Also print to console
        print("\n".join(block))
        print(f"(Also saved target as {target_name})")

    # -----------------------------
    # Build PDF Report
    # -----------------------------
    pdf_path = os.path.join(out_dir, "report.pdf")
    with PdfPages(pdf_path) as pdf:
        # Cover page
        cover_lines = [
            "AI Melody Improviser – Multi-Style Report",
            f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            f"Target Melody (user): {' '.join(user_melody)}",
            f"Runs per style: {runs}, Generations: {GENERATIONS}, Population: {POP_SIZE}",
            "",
            "This report compares GA outputs across Soft, Rap, and Classical styles",
            "against the same target melody, including run logs, fitness graphs,",
            "pitch-contour comparisons, and a summary table.",
        ]
        fig = page_title_fig(cover_lines)
        pdf.savefig(fig); plt.close(fig)

        # Text log page (emoji-free)
        log_text = "\n\n".join(full_log_lines)
        fig = monospaced_text_page(log_text)
        pdf.savefig(fig); plt.close(fig)

        # Per-style fitness pages (mean ± std)
        for style in styles:
            res = results[style]
            fig, ax = plt.subplots(figsize=(10, 5))
            plot_fitness_band(ax, res["best_traces_mean"], res["best_traces_std"], f"{style.title()} Best (mean ± std)")
            plot_fitness_band(ax, res["avg_traces_mean"], res["avg_traces_std"], f"{style.title()} Avg (mean ± std)")
            ax.set_title(f"Fitness Evolution – {style.title()} (scale: {res['scale']}, tempo: {res['tempo']} BPM)")
            ax.set_xlabel("Generation")
            ax.set_ylabel("Fitness")
            ax.grid(True)
            ax.legend()
            pdf.savefig(fig); plt.close(fig)

        # Per-style pitch contour comparison (Target vs Generated)
        for style in styles:
            res = results[style]
            fig, ax = plt.subplots(figsize=(10, 4))
            plot_pitch_contour(ax, res["target"], res["best_melody"], f"{style.title()} – Pitch Contour")
            pdf.savefig(fig); plt.close(fig)

        # Summary table page
        table_rows = []
        for style in styles:
            all_bs = results[style]["all_best_scores"]
            table_rows.append([
                style.title(),
                f"{mean(all_bs):.2f}",
                f"{max(all_bs):.2f}",
                str(results[style]["tempo"]),
                results[style]["scale"],
            ])
        fig = summary_table_page(table_rows, title="Summary Comparison (Avg vs Best)")
        pdf.savefig(fig); plt.close(fig)

        # Final summary text page
        summary_lines = ["Scores (detail)"]
        for style in styles:
            bs = results[style]["best_score"]
            all_bs = results[style]["all_best_scores"]
            summary_lines.append(
                f"{style.title()}: best={bs}, runs_mean={mean(all_bs):.2f}, runs_std={pstdev(all_bs):.2f}"
            )
        fig = page_title_fig(["Summary"] + summary_lines + ["", f"Output folder: {out_dir}"])
        pdf.savefig(fig); plt.close(fig)

    print(f"\nPDF report saved to: {pdf_path}")
    print(f"All outputs saved in folder: {out_dir}")
    print("Done ✓")

if __name__ == "__main__":
    main()



=== AI Melody Improviser – Single Target, Multi-Style Report ===

Enter target melody (space- or comma-separated, e.g. C C G G A A G F F E E D D C): C D E F G G A G F E D C
Number of GA runs per style (e.g., 5 or 10): 5
Generations per run (default 120): 200
Population size (default 60): 100

(Press Enter to keep defaults) Tempo defaults -> Soft: 90, Rap: 120, Classical: 72
Tempo for soft BPM (default 90): 80
Tempo for rap BPM (default 120): 140
Tempo for classical BPM (default 72): 100

Available scales: C major, A minor, C minor
Scale to use for ALL styles (blank = per-style defaults): C Major , C Minor, A minor
Scale not found; using per-style defaults.

>>> Running 5 GA runs for Soft Style...
♪ Running GA for Soft Style...
Saved Soft generated melody to Soft_generated_135308.mid
♪ Target Melody: ['C', 'D', 'E', 'F', 'G', 'G', 'A', 'G', 'F', 'E', 'D', 'C']
♪ Best Generated Melody: ['D', 'D', 'E', 'F', 'G', 'G', 'G', 'G', 'F', 'E', 'D', 'D']
✓ Fitness Score: 63.5
(Also saved target 

In [14]:
import glob
import subprocess
from pydub import AudioSegment

# Find all generated MIDI files
midi_files = glob.glob("**/*.mid", recursive=True)

for midi_file in midi_files:
    wav_file = midi_file.replace(".mid", ".wav")
    mp3_file = midi_file.replace(".mid", ".mp3")

    print(f"Converting {midi_file}...")

    # MIDI -> WAV
    subprocess.run([
        "fluidsynth",
        "-ni",
        "TimGM6mb.sf2",
        midi_file,
        "-F",
        wav_file,
        "-r",
        "44100"
    ], check=True)

    # WAV -> MP3
    AudioSegment.from_wav(wav_file).export(mp3_file, format="mp3")

    print(f"Created: {mp3_file}")

print("\n✅ All MIDI files converted to MP3.")

Converting aimelody_report_20260717_135307/Classical_target_135311.mid...


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Created: aimelody_report_20260717_135307/Classical_target_135311.mp3
Converting aimelody_report_20260717_135307/Soft_generated_135308.mid...
Created: aimelody_report_20260717_135307/Soft_generated_135308.mp3
Converting aimelody_report_20260717_135307/Rap_target_135309.mid...
Created: aimelody_report_20260717_135307/Rap_target_135309.mp3
Converting aimelody_report_20260717_135307/Soft_target_135308.mid...
Created: aimelody_report_20260717_135307/Soft_target_135308.mp3
Converting aimelody_report_20260717_135307/Rap_generated_135309.mid...
Created: aimelody_report_20260717_135307/Rap_generated_135309.mp3
Converting aimelody_report_20260717_135307/Classical_generated_135311.mid...
Created: aimelody_report_20260717_135307/Classical_generated_135311.mp3
Converting aimelody_report_20260717_132553/Soft_target_132554.mid...
Created: aimelody_report_20260717_132553/Soft_target_132554.mp3
Converting aimelody_report_20260717_132553/Rap_generated_132555.mid...
Created: aimelody_report_20260717_1325